In [2]:
# ============================================================
# CELL X — CROSS-DAY UNIT MATCHING (PAIRWISE)
# ============================================================
# For each pair of sessions, computes match scores between all
# units, applies conservative hard filters, ranks candidates, and
# enforces bijection (mutual best-match).
#
# Hard filters:
#   - Centroid distance ≤ MAX_CENTROID_DIST_UM
#   - Log firing rate ratio within ±MAX_LOG_RATE_DIFF
#   - Waveform correlation ≥ MIN_WAVEFORM_CORR
#
# Soft score combines:
#   - Waveform correlation
#   - ACG correlation
#   - Firing rate similarity
#   - Width/trough similarity
#   - Burst stats similarity (if available)
#   - Reach PSTH similarity (if both modulated)
#
# Output: matched_pairs_{sessionA}_{sessionB}.npz per session pair
# ============================================================
import json
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr
from itertools import combinations

print("\n===== CELL X — CROSS-DAY UNIT MATCHING =====")

# ------------------------------------------------------------
# USER EDIT — sessions to compare
# ------------------------------------------------------------
SESSION_ANALYSIS_FOLDERS = [
    r"G:\Kevin\2026-03-04_08-31-33\Record Node 101\2026-03-04_08-31-33_experiment1_recording1_analysis",
    r"G:\Kevin\2026-03-05_09-27-03\Record Node 101\2026-03-05_09-27-03_experiment2_recording1_analysis",
    r"G:\Kevin\2026-03-09_14-29-25\Record Node 114\2026-03-09_14-29-25_experiment2_recording1_analysis",
]

# Conservative hard filter thresholds
MAX_CENTROID_DIST_UM    = 30.0   # spatial proximity required
MAX_LOG_RATE_DIFF       = 0.3    # log(rate_A / rate_B) within ±0.3
MIN_WAVEFORM_CORR       = 0.85   # waveform shape similarity
MIN_ACG_CORR            = 0.5    # ACG shape similarity (softer filter)

# Soft scoring weights
W_WAVEFORM      = 3.0
W_ACG           = 2.0
W_RATE          = 1.0
W_WIDTH         = 1.0
W_BURST         = 1.0
W_REACH         = 2.0

# Score threshold (out of max possible)
MIN_MATCH_SCORE = 0.65   # 65% of max combined score required

# Output directory: top-level matches/ folder shared across sessions
OUTPUT_BASE = r"G:\Kevin\unit_matching"
# ------------------------------------------------------------

output_base = Path(OUTPUT_BASE)
output_base.mkdir(exist_ok=True, parents=True)

# ------------------------------------------------------------
# LOAD FINGERPRINTS FROM EACH SESSION
# ------------------------------------------------------------
def load_fingerprints(session_folder):
    fp_path = Path(session_folder) / "fingerprints" / "unit_fingerprints.npz"
    if not fp_path.exists():
        raise FileNotFoundError(f"No fingerprint at {fp_path}")
    fp = dict(np.load(fp_path, allow_pickle=True))
    fp["session_name"] = Path(session_folder).name
    fp["session_folder"] = str(session_folder)
    return fp

session_fps = []
for sf in SESSION_ANALYSIS_FOLDERS:
    print(f"\nLoading: {Path(sf).name}")
    fp = load_fingerprints(sf)
    print(f"  Units: {len(fp['unit_ids'])}")
    print(f"  Has bursts: {bool(fp['has_bursts'])}")
    print(f"  Has reach: {bool(fp['has_reach'])}")
    session_fps.append(fp)

# ------------------------------------------------------------
# MATCHING FUNCTIONS
# ------------------------------------------------------------
def centroid_distance(fp_a, fp_b, i, j):
    """Euclidean distance between two unit centroids."""
    dx = fp_a["centroid_x"][i] - fp_b["centroid_x"][j]
    dy = fp_a["centroid_y"][i] - fp_b["centroid_y"][j]
    return float(np.sqrt(dx**2 + dy**2))

def log_rate_diff(fp_a, fp_b, i, j):
    """Absolute log-ratio of firing rates."""
    rate_a = max(fp_a["firing_rate"][i], 1e-6)
    rate_b = max(fp_b["firing_rate"][j], 1e-6)
    return abs(np.log(rate_a / rate_b))

def waveform_correlation(fp_a, fp_b, i, j):
    """Pearson r between waveforms on each unit's best channel."""
    wa = fp_a["waveform_best"][i]
    wb = fp_b["waveform_best"][j]
    if wa.std() == 0 or wb.std() == 0:
        return 0.0
    r, _ = pearsonr(wa, wb)
    return float(r)

def acg_correlation(fp_a, fp_b, i, j):
    """Pearson r between autocorrelograms."""
    ac_a = fp_a["acg"][i]
    ac_b = fp_b["acg"][j]
    if ac_a.std() == 0 or ac_b.std() == 0:
        return 0.0
    r, _ = pearsonr(ac_a, ac_b)
    return float(r)

def rate_similarity(fp_a, fp_b, i, j):
    """1 - normalized log-rate difference (0 to 1)."""
    diff = log_rate_diff(fp_a, fp_b, i, j)
    return max(0.0, 1 - diff / MAX_LOG_RATE_DIFF)

def width_similarity(fp_a, fp_b, i, j):
    """1 if widths match perfectly, decays to 0 at 1ms difference."""
    wa = fp_a["width_ms"][i]
    wb = fp_b["width_ms"][j]
    diff = abs(wa - wb)
    return max(0.0, 1 - diff / 1.0)

def burst_similarity(fp_a, fp_b, i, j):
    """Similarity of burst fraction. Only useful if both sessions have bursts."""
    if not (fp_a["has_bursts"] and fp_b["has_bursts"]):
        return None
    ba = fp_a["burst_fraction"][i]
    bb = fp_b["burst_fraction"][j]
    diff = abs(ba - bb)
    return max(0.0, 1 - diff / 0.05)  # decays over 5% burst fraction range

def reach_similarity(fp_a, fp_b, i, j):
    """Pearson r between reach PSTH z-traces (only if both reach-modulated AND same length)."""
    if not (fp_a["has_reach"] and fp_b["has_reach"]):
        return None
    if not (fp_a["reach_modulated"][i] and fp_b["reach_modulated"][j]):
        return None
    z_a = fp_a["reach_z_mean"][i]
    z_b = fp_b["reach_z_mean"][j]
    # Sessions may have used different windows/bins — skip if lengths differ
    if len(z_a) != len(z_b):
        return None
    if z_a.std() == 0 or z_b.std() == 0:
        return 0.0
    r, _ = pearsonr(z_a, z_b)
    return max(0.0, float(r))

def compute_match_score(fp_a, fp_b, i, j):
    """Combined match score, normalized to 0-1."""
    waveform_r = waveform_correlation(fp_a, fp_b, i, j)
    acg_r = acg_correlation(fp_a, fp_b, i, j)
    rate_s = rate_similarity(fp_a, fp_b, i, j)
    width_s = width_similarity(fp_a, fp_b, i, j)
    burst_s = burst_similarity(fp_a, fp_b, i, j)
    reach_s = reach_similarity(fp_a, fp_b, i, j)

    weighted_sum = 0.0
    max_sum = 0.0

    # Required components
    weighted_sum += W_WAVEFORM * waveform_r
    max_sum += W_WAVEFORM

    weighted_sum += W_ACG * acg_r
    max_sum += W_ACG

    weighted_sum += W_RATE * rate_s
    max_sum += W_RATE

    weighted_sum += W_WIDTH * width_s
    max_sum += W_WIDTH

    # Optional components
    if burst_s is not None:
        weighted_sum += W_BURST * burst_s
        max_sum += W_BURST

    if reach_s is not None:
        weighted_sum += W_REACH * reach_s
        max_sum += W_REACH

    score = weighted_sum / max_sum if max_sum > 0 else 0.0
    return {
        "score": score,
        "waveform_r": waveform_r,
        "acg_r": acg_r,
        "rate_similarity": rate_s,
        "width_similarity": width_s,
        "burst_similarity": burst_s,
        "reach_similarity": reach_s,
    }

# ------------------------------------------------------------
# PAIRWISE MATCHING
# ------------------------------------------------------------
def match_session_pair(fp_a, fp_b):
    """Compute all candidate scores, apply filters, return mutual best matches."""
    n_a = len(fp_a["unit_ids"])
    n_b = len(fp_b["unit_ids"])

    score_matrix = np.zeros((n_a, n_b))
    filter_passed = np.zeros((n_a, n_b), dtype=bool)
    all_metrics = {}

    print(f"  Computing scores for {n_a} × {n_b} = {n_a * n_b} pairs...")

    for i in range(n_a):
        for j in range(n_b):
            # Hard filters
            dist = centroid_distance(fp_a, fp_b, i, j)
            if dist > MAX_CENTROID_DIST_UM:
                continue

            rate_diff = log_rate_diff(fp_a, fp_b, i, j)
            if rate_diff > MAX_LOG_RATE_DIFF:
                continue

            wave_r = waveform_correlation(fp_a, fp_b, i, j)
            if wave_r < MIN_WAVEFORM_CORR:
                continue

            acg_r = acg_correlation(fp_a, fp_b, i, j)
            if acg_r < MIN_ACG_CORR:
                continue

            # Passed hard filters; compute full score
            metrics = compute_match_score(fp_a, fp_b, i, j)
            score_matrix[i, j] = metrics["score"]
            filter_passed[i, j] = True
            all_metrics[(i, j)] = metrics

    # Find mutual best matches (bijection)
    matches = []
    for i in range(n_a):
        if not filter_passed[i].any():
            continue
        best_j = int(np.argmax(score_matrix[i]))
        best_score = score_matrix[i, best_j]
        if best_score < MIN_MATCH_SCORE:
            continue

        # Reciprocity: is this i also the best for j?
        best_i_for_j = int(np.argmax(score_matrix[:, best_j]))
        if best_i_for_j != i:
            continue

        metrics = all_metrics[(i, best_j)]
        matches.append({
            "session_a_unit": int(fp_a["unit_ids"][i]),
            "session_b_unit": int(fp_b["unit_ids"][best_j]),
            "session_a_idx": int(i),
            "session_b_idx": int(best_j),
            "score": float(best_score),
            "centroid_distance_um": centroid_distance(fp_a, fp_b, i, best_j),
            "log_rate_diff": log_rate_diff(fp_a, fp_b, i, best_j),
            "waveform_r": metrics["waveform_r"],
            "acg_r": metrics["acg_r"],
            "rate_similarity": metrics["rate_similarity"],
            "width_similarity": metrics["width_similarity"],
            "burst_similarity": (metrics["burst_similarity"]
                                  if metrics["burst_similarity"] is not None
                                  else -1),
            "reach_similarity": (metrics["reach_similarity"]
                                  if metrics["reach_similarity"] is not None
                                  else -1),
        })

    return matches, score_matrix, filter_passed

# ------------------------------------------------------------
# RUN ALL PAIRWISE COMPARISONS
# ------------------------------------------------------------
all_results = {}

for idx_a, idx_b in combinations(range(len(session_fps)), 2):
    fp_a = session_fps[idx_a]
    fp_b = session_fps[idx_b]
    name_a = fp_a["session_name"]
    name_b = fp_b["session_name"]
    pair_key = f"{name_a}__vs__{name_b}"

    print(f"\n--- Matching {name_a} ↔ {name_b} ---")
    matches, score_matrix, filter_passed = match_session_pair(fp_a, fp_b)

    n_passed_filter = filter_passed.sum()
    print(f"  Pairs passing hard filters: {n_passed_filter}")
    print(f"  Mutual best matches: {len(matches)}")
    if len(matches):
        scores = [m["score"] for m in matches]
        print(f"  Score range: {min(scores):.3f} → {max(scores):.3f}, "
              f"median {np.median(scores):.3f}")

    # Save
    out_path = output_base / f"matched_pairs_{pair_key}.npz"
    if matches:
        np.savez(out_path,
                 session_a_units=np.array([m["session_a_unit"] for m in matches]),
                 session_b_units=np.array([m["session_b_unit"] for m in matches]),
                 session_a_idx=np.array([m["session_a_idx"] for m in matches]),
                 session_b_idx=np.array([m["session_b_idx"] for m in matches]),
                 scores=np.array([m["score"] for m in matches]),
                 centroid_distances=np.array([m["centroid_distance_um"] for m in matches]),
                 log_rate_diffs=np.array([m["log_rate_diff"] for m in matches]),
                 waveform_rs=np.array([m["waveform_r"] for m in matches]),
                 acg_rs=np.array([m["acg_r"] for m in matches]),
                 rate_similarities=np.array([m["rate_similarity"] for m in matches]),
                 width_similarities=np.array([m["width_similarity"] for m in matches]),
                 burst_similarities=np.array([m["burst_similarity"] for m in matches]),
                 reach_similarities=np.array([m["reach_similarity"] for m in matches]),
                 session_a_name=np.array(name_a),
                 session_b_name=np.array(name_b),
                 session_a_folder=np.array(fp_a["session_folder"]),
                 session_b_folder=np.array(fp_b["session_folder"]),
                 )
        print(f"  Saved → {out_path.name}")
    else:
        print(f"  No matches found.")

    all_results[pair_key] = {
        "n_matches": len(matches),
        "matches": matches,
    }

# ------------------------------------------------------------
# SUMMARY TABLE
# ------------------------------------------------------------
print("\n===== MATCHING SUMMARY =====")
print(f"{'Comparison':<60} {'Matches':>10} {'Match %':>10}")
print("-" * 85)
for pair_key, result in all_results.items():
    name_a, name_b = pair_key.split("__vs__")
    n_a = len([f for f in session_fps if f["session_name"] == name_a][0]["unit_ids"])
    n_b = len([f for f in session_fps if f["session_name"] == name_b][0]["unit_ids"])
    match_pct = 100 * 2 * result["n_matches"] / (n_a + n_b)
    print(f"{name_a[:25]:<30}{name_b[:25]:<30} {result['n_matches']:>10d} {match_pct:>9.1f}%")

# Save overall summary
with open(output_base / "matching_summary.json", "w") as f:
    json.dump({
        "criteria": {
            "max_centroid_dist_um": MAX_CENTROID_DIST_UM,
            "max_log_rate_diff": MAX_LOG_RATE_DIFF,
            "min_waveform_corr": MIN_WAVEFORM_CORR,
            "min_acg_corr": MIN_ACG_CORR,
            "min_match_score": MIN_MATCH_SCORE,
        },
        "weights": {
            "waveform": W_WAVEFORM,
            "acg": W_ACG,
            "rate": W_RATE,
            "width": W_WIDTH,
            "burst": W_BURST,
            "reach": W_REACH,
        },
        "results": {k: v["n_matches"] for k, v in all_results.items()},
    }, f, indent=2)

print("\n===== CELL X COMPLETE =====\n")


===== CELL X — CROSS-DAY UNIT MATCHING =====

Loading: 2026-03-04_08-31-33_experiment1_recording1_analysis
  Units: 100
  Has bursts: True
  Has reach: True

Loading: 2026-03-05_09-27-03_experiment2_recording1_analysis
  Units: 192
  Has bursts: True
  Has reach: True

Loading: 2026-03-09_14-29-25_experiment2_recording1_analysis
  Units: 185
  Has bursts: True
  Has reach: True

--- Matching 2026-03-04_08-31-33_experiment1_recording1_analysis ↔ 2026-03-05_09-27-03_experiment2_recording1_analysis ---
  Computing scores for 100 × 192 = 19200 pairs...
  Pairs passing hard filters: 158
  Mutual best matches: 66
  Score range: 0.746 → 0.989, median 0.909
  Saved → matched_pairs_2026-03-04_08-31-33_experiment1_recording1_analysis__vs__2026-03-05_09-27-03_experiment2_recording1_analysis.npz

--- Matching 2026-03-04_08-31-33_experiment1_recording1_analysis ↔ 2026-03-09_14-29-25_experiment2_recording1_analysis ---
  Computing scores for 100 × 185 = 18500 pairs...
  Pairs passing hard filters: 

In [10]:
# ============================================================
# CELL Y — VISUAL VALIDATION OF MATCHED UNIT PAIRS (PATCHED)
# ============================================================
# Same as before but handles cross-session PSTHs with different
# window/bin settings (plots them on bin-index x-axis with warning).
# ============================================================
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
from itertools import combinations

print("\n===== CELL Y — MATCHED PAIR VISUALIZATION =====")

# ------------------------------------------------------------
# USER EDIT
# ------------------------------------------------------------
SESSION_ANALYSIS_FOLDERS = [
    r"G:\Kevin\2026-03-04_08-31-33\Record Node 101\2026-03-04_08-31-33_experiment1_recording1_analysis",
    r"G:\Kevin\2026-03-05_09-27-03\Record Node 101\2026-03-05_09-27-03_experiment2_recording1_analysis",
    r"G:\Kevin\2026-03-09_14-29-25\Record Node 114\2026-03-09_14-29-25_experiment2_recording1_analysis",
]
OUTPUT_BASE = r"G:\Kevin\unit_matching"
N_TOP_TO_PLOT = 50
# ------------------------------------------------------------

output_base = Path(OUTPUT_BASE)

# ------------------------------------------------------------
# LOAD FINGERPRINTS
# ------------------------------------------------------------
session_fps = []
for sf in SESSION_ANALYSIS_FOLDERS:
    fp_path = Path(sf) / "fingerprints" / "unit_fingerprints.npz"
    fp = dict(np.load(fp_path, allow_pickle=True))
    fp["session_name"] = Path(sf).name
    fp["session_folder"] = str(sf)
    session_fps.append(fp)
print(f"Loaded {len(session_fps)} session fingerprints")

# ------------------------------------------------------------
# PLOT ONE MATCHED PAIR
# ------------------------------------------------------------
def plot_matched_pair(fp_a, fp_b, idx_a, idx_b, match_info, pair_idx, n_pairs):
    """Produce side-by-side comparison of two matched units."""
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(3, 4, hspace=0.4, wspace=0.35)

    unit_a = int(fp_a["unit_ids"][idx_a])
    unit_b = int(fp_b["unit_ids"][idx_b])

    title = (f"Match {pair_idx+1}/{n_pairs}: "
             f"{fp_a['session_name'][:20]} unit {unit_a} ↔ "
             f"{fp_b['session_name'][:20]} unit {unit_b}\n"
             f"score={match_info['score']:.3f}  |  "
             f"centroid_dist={match_info['centroid_distance_um']:.1f} µm  |  "
             f"waveform_r={match_info['waveform_r']:.3f}  |  "
             f"acg_r={match_info['acg_r']:.3f}")
    fig.suptitle(title, fontsize=10, y=0.99)

    # --- Waveforms ---
    ax_wf = fig.add_subplot(gs[0, :2])
    wf_a = fp_a["waveform_best"][idx_a]
    wf_b = fp_b["waveform_best"][idx_b]
    t_ms_a = np.arange(len(wf_a)) / fp_a["sample_rate_hz"] * 1000
    t_ms_b = np.arange(len(wf_b)) / fp_b["sample_rate_hz"] * 1000
    ax_wf.plot(t_ms_a, wf_a, color="C3", lw=1.5,
               label=f"{fp_a['session_name'][:15]}: unit {unit_a}")
    ax_wf.plot(t_ms_b, wf_b, color="C0", lw=1.5,
               label=f"{fp_b['session_name'][:15]}: unit {unit_b}")
    ax_wf.axhline(0, color="gray", lw=0.5)
    ax_wf.set_title("Waveforms (best channel)", fontsize=10)
    ax_wf.set_xlabel("Time (ms)")
    ax_wf.set_ylabel("Amplitude")
    ax_wf.legend(fontsize=8)

    # --- Autocorrelograms ---
    ax_acg = fig.add_subplot(gs[0, 2:])
    ac_a = fp_a["acg"][idx_a]
    ac_b = fp_b["acg"][idx_b]
    bin_size_ms = 1.0
    lags_a = np.arange(len(ac_a)) * bin_size_ms
    lags_b = np.arange(len(ac_b)) * bin_size_ms
    ax_acg.plot(lags_a, ac_a, color="C3", lw=1.5,
                label=f"{fp_a['session_name'][:15]}: unit {unit_a}")
    ax_acg.plot(lags_b, ac_b, color="C0", lw=1.5,
                label=f"{fp_b['session_name'][:15]}: unit {unit_b}")
    ax_acg.set_xlabel("Lag (ms)")
    ax_acg.set_ylabel("Probability")
    ax_acg.set_title(f"Autocorrelograms (r={match_info['acg_r']:.3f})", fontsize=10)
    ax_acg.legend(fontsize=8)

    # --- Reach PSTH (handles mismatched lengths) ---
    ax_reach = fig.add_subplot(gs[1, :2])
    if fp_a["has_reach"] and fp_b["has_reach"]:
        a_mod = fp_a["reach_modulated"][idx_a]
        b_mod = fp_b["reach_modulated"][idx_b]
        z_a = fp_a["reach_z_mean"][idx_a]
        z_b = fp_b["reach_z_mean"][idx_b]

        # Try to load centers from session A; fall back to bin indices
        centers_path_a = Path(fp_a["session_folder"]) / "qc" / "reach_good_in_trial_arrays.npz"
        centers_a = (np.load(centers_path_a)["centers"]
                     if centers_path_a.exists() else np.arange(len(z_a)))
        centers_path_b = Path(fp_b["session_folder"]) / "qc" / "reach_good_in_trial_arrays.npz"
        centers_b = (np.load(centers_path_b)["centers"]
                     if centers_path_b.exists() else np.arange(len(z_b)))

        if len(z_a) != len(z_b):
            # Mismatched window/bin settings — use bin indices
            ax_reach.plot(np.arange(len(z_a)), z_a, color="C3", lw=1.5,
                          label=f"{fp_a['session_name'][:15]} "
                                f"({'mod' if a_mod else 'unmod'}, peak {fp_a['reach_peak_z'][idx_a]:.1f}z)")
            ax_reach.plot(np.arange(len(z_b)), z_b, color="C0", lw=1.5,
                          label=f"{fp_b['session_name'][:15]} "
                                f"({'mod' if b_mod else 'unmod'}, peak {fp_b['reach_peak_z'][idx_b]:.1f}z)")
            ax_reach.set_xlabel("Bin index")
            ax_reach.text(0.02, 0.95, "⚠ different windows; x-axis is bin index",
                          transform=ax_reach.transAxes, fontsize=7, color="orange",
                          verticalalignment="top")
        else:
            ax_reach.plot(centers_a, z_a, color="C3", lw=1.5,
                          label=f"{fp_a['session_name'][:15]} "
                                f"({'mod' if a_mod else 'unmod'}, peak {fp_a['reach_peak_z'][idx_a]:.1f}z)")
            ax_reach.plot(centers_b, z_b, color="C0", lw=1.5,
                          label=f"{fp_b['session_name'][:15]} "
                                f"({'mod' if b_mod else 'unmod'}, peak {fp_b['reach_peak_z'][idx_b]:.1f}z)")
            ax_reach.axvline(0, color="black", lw=0.5, linestyle="--")
            ax_reach.set_xlabel("Time from reach (s)")

        ax_reach.axhline(0, color="gray", lw=0.5)
        reach_r = match_info["reach_similarity"]
        title_str = "Reach-aligned z-PSTH"
        if reach_r > 0:
            title_str += f" (r={reach_r:.3f})"
        elif reach_r == -1:
            title_str += " (not comparable across sessions)"
        ax_reach.set_title(title_str, fontsize=10)
        ax_reach.set_ylabel("z-score")
        ax_reach.legend(fontsize=8)
    else:
        ax_reach.text(0.5, 0.5, "No reach data available",
                      ha="center", va="center", transform=ax_reach.transAxes)
        ax_reach.set_xticks([]); ax_reach.set_yticks([])

    # --- Channel position overlay ---
    ax_pos = fig.add_subplot(gs[1, 2:])
    ch_pos = fp_a["channel_positions"]
    ax_pos.scatter(ch_pos[:, 0], ch_pos[:, 1], s=8, color="lightgray",
                   alpha=0.6, label="channels")
    ax_pos.scatter([fp_a["centroid_x"][idx_a]],
                   [fp_a["centroid_y"][idx_a]],
                   s=200, marker="o", color="C3", edgecolor="black",
                   label=f"{fp_a['session_name'][:15]}: ch {fp_a['best_ch'][idx_a]}")
    ax_pos.scatter([fp_b["centroid_x"][idx_b]],
                   [fp_b["centroid_y"][idx_b]],
                   s=200, marker="^", color="C0", edgecolor="black",
                   label=f"{fp_b['session_name'][:15]}: ch {fp_b['best_ch'][idx_b]}")
    ax_pos.set_xlabel("X (µm)")
    ax_pos.set_ylabel("Y (µm)")
    ax_pos.set_title(f"Probe position (dist={match_info['centroid_distance_um']:.1f} µm)",
                     fontsize=10)
    ax_pos.legend(fontsize=8)
    ax_pos.set_aspect("equal")

    centroid_y_mean = (fp_a["centroid_y"][idx_a] + fp_b["centroid_y"][idx_b]) / 2
    centroid_x_mean = (fp_a["centroid_x"][idx_a] + fp_b["centroid_x"][idx_b]) / 2
    ax_pos.set_xlim(centroid_x_mean - 80, centroid_x_mean + 80)
    ax_pos.set_ylim(centroid_y_mean - 100, centroid_y_mean + 100)

    # --- Numerical comparison table ---
    ax_stats = fig.add_subplot(gs[2, :])
    ax_stats.axis("off")

    rows = [
        ("Metric", f"{fp_a['session_name'][:18]}", f"{fp_b['session_name'][:18]}", "Δ / similarity"),
        ("Unit ID", str(unit_a), str(unit_b), "—"),
        ("Best ch", str(fp_a["best_ch"][idx_a]), str(fp_b["best_ch"][idx_b]),
         f"Δ = {int(fp_a['best_ch'][idx_a]) - int(fp_b['best_ch'][idx_b])}"),
        ("Firing rate (Hz)", f"{fp_a['firing_rate'][idx_a]:.2f}",
         f"{fp_b['firing_rate'][idx_b]:.2f}",
         f"log-ratio = {match_info['log_rate_diff']:.3f}"),
        ("Waveform width (ms)", f"{fp_a['width_ms'][idx_a]:.2f}",
         f"{fp_b['width_ms'][idx_b]:.2f}",
         f"sim = {match_info['width_similarity']:.3f}"),
        ("Peak-to-peak amp", f"{fp_a['peak_to_peak_amp'][idx_a]:.2f}",
         f"{fp_b['peak_to_peak_amp'][idx_b]:.2f}", "—"),
        ("ISI CV", f"{fp_a['isi_cv'][idx_a]:.2f}",
         f"{fp_b['isi_cv'][idx_b]:.2f}", "—"),
    ]
    if fp_a["has_bursts"] and fp_b["has_bursts"]:
        rows.append(("Burst fraction (%)", f"{fp_a['burst_fraction'][idx_a]*100:.2f}",
                     f"{fp_b['burst_fraction'][idx_b]*100:.2f}",
                     f"sim = {match_info['burst_similarity']:.3f}"
                     if match_info['burst_similarity'] >= 0 else "—"))
        rows.append(("Burst rate (Hz)", f"{fp_a['burst_rate'][idx_a]:.3f}",
                     f"{fp_b['burst_rate'][idx_b]:.3f}", "—"))

    table = ax_stats.table(cellText=rows[1:],
                            colLabels=rows[0],
                            loc="center",
                            cellLoc="center",
                            colColours=["lightgray"] * 4)
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.6)

    return fig

# ------------------------------------------------------------
# PROCESS EACH PAIR
# ------------------------------------------------------------
n_total_plots = 0
for idx_a, idx_b in combinations(range(len(session_fps)), 2):
    fp_a = session_fps[idx_a]
    fp_b = session_fps[idx_b]
    name_a = fp_a["session_name"]
    name_b = fp_b["session_name"]
    pair_key = f"{name_a}__vs__{name_b}"

    matches_path = output_base / f"matched_pairs_{pair_key}.npz"
    if not matches_path.exists():
        print(f"\nNo matches file for {pair_key}")
        continue

    matches_data = np.load(matches_path, allow_pickle=True)
    n_matches = len(matches_data["scores"])
    print(f"\n--- {pair_key} ---")
    print(f"  Total matches: {n_matches}")

    if n_matches == 0:
        continue

    order = np.argsort(matches_data["scores"])[::-1][:N_TOP_TO_PLOT]

    pdf_path = output_base / f"validation_{pair_key}.pdf"
    with PdfPages(pdf_path) as pdf:
        for plot_i, m_idx in enumerate(order):
            match_info = {
                "score": float(matches_data["scores"][m_idx]),
                "centroid_distance_um": float(matches_data["centroid_distances"][m_idx]),
                "log_rate_diff": float(matches_data["log_rate_diffs"][m_idx]),
                "waveform_r": float(matches_data["waveform_rs"][m_idx]),
                "acg_r": float(matches_data["acg_rs"][m_idx]),
                "rate_similarity": float(matches_data["rate_similarities"][m_idx]),
                "width_similarity": float(matches_data["width_similarities"][m_idx]),
                "burst_similarity": float(matches_data["burst_similarities"][m_idx]),
                "reach_similarity": float(matches_data["reach_similarities"][m_idx]),
            }
            idx_a_match = int(matches_data["session_a_idx"][m_idx])
            idx_b_match = int(matches_data["session_b_idx"][m_idx])

            fig = plot_matched_pair(fp_a, fp_b, idx_a_match, idx_b_match,
                                    match_info, plot_i, min(n_matches, N_TOP_TO_PLOT))
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            n_total_plots += 1

    print(f"  Saved → {pdf_path.name}")

print(f"\nTotal validation plots: {n_total_plots}")
print("\n===== CELL Y COMPLETE =====\n")


===== CELL Y — MATCHED PAIR VISUALIZATION =====
Loaded 3 session fingerprints

--- 2026-03-04_08-31-33_experiment1_recording1_analysis__vs__2026-03-05_09-27-03_experiment2_recording1_analysis ---
  Total matches: 66
  Saved → validation_2026-03-04_08-31-33_experiment1_recording1_analysis__vs__2026-03-05_09-27-03_experiment2_recording1_analysis.pdf

--- 2026-03-04_08-31-33_experiment1_recording1_analysis__vs__2026-03-09_14-29-25_experiment2_recording1_analysis ---
  Total matches: 44
  Saved → validation_2026-03-04_08-31-33_experiment1_recording1_analysis__vs__2026-03-09_14-29-25_experiment2_recording1_analysis.pdf

--- 2026-03-05_09-27-03_experiment2_recording1_analysis__vs__2026-03-09_14-29-25_experiment2_recording1_analysis ---
  Total matches: 70
  Saved → validation_2026-03-05_09-27-03_experiment2_recording1_analysis__vs__2026-03-09_14-29-25_experiment2_recording1_analysis.pdf

Total validation plots: 144

===== CELL Y COMPLETE =====

